# Preprocessing and exploratory analysis

This notebook applies one preprocessing contract to gold and unlabeled reviews, creates privacy-safe analytical text, records outlier diagnostics, and produces source-specific exploratory summaries.


In [ ]:
import json
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import (
    ArrayType, DoubleType, IntegerType, StringType, StructField, StructType
)

BASE_DIR = Path("/kaggle/working/bangla_fake_review")
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
EDA_DIR = BASE_DIR / "eda"
for directory in (AUDIT_DIR, EDA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("BanglaFakeReview_Preprocessing")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

train = spark.read.parquet(str(PROCESSED_DIR / "bfrd_train.parquet"))
validation = spark.read.parquet(str(PROCESSED_DIR / "bfrd_validation.parquet"))
test = spark.read.parquet(str(PROCESSED_DIR / "bfrd_test_locked.parquet"))
banglish = (
    spark.read.parquet(str(PROCESSED_DIR / "banglish_unlabeled.parquet"))
    .drop("reply")
)


In [ ]:
URL_PATTERN = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
PHONE_PATTERN = re.compile(r"(?:\+?88)?01[3-9]\d{8}")
PRICE_PATTERN = re.compile(
    r"(?:৳|tk|bdt)\s*[০-৯\d]+(?:[.,][০-৯\d]+)?|"
    r"[০-৯\d]+(?:[.,][০-৯\d]+)?\s*(?:৳|tk|bdt)",
    re.IGNORECASE,
)
HASHTAG_PATTERN = re.compile(r"#[\w\u0980-\u09ff]+")
MENTION_PATTERN = re.compile(r"@[\w\u0980-\u09ff]+")
REPEATED_PATTERN = re.compile(r"(.)\1{2,}", re.UNICODE)
CONTROL_PATTERN = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
TOKEN_PATTERN = re.compile(r"[\u0980-\u09ff]+|[A-Za-z]+|[০-৯\d]+", re.UNICODE)

STOPWORD_VERSION = "bn-en-2026-08-24"
BANGLA_STOPWORDS = {
    "আমি", "আমরা", "তুমি", "আপনি", "সে", "তিনি", "তারা", "এই", "এটা", "ওটা",
    "এবং", "আর", "বা", "কিন্তু", "যে", "কি", "না", "হয়", "হয়", "ছিল", "আছে",
    "থেকে", "জন্য", "সাথে", "একটি", "একটা", "এর", "তে", "কে", "ও", "তো",
}
ENGLISH_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from",
    "has", "have", "he", "i", "in", "is", "it", "of", "on", "or", "that",
    "the", "this", "to", "was", "we", "were", "with", "you",
}
PROMOTIONAL_TERMS = {
    "discount", "offer", "sale", "deal", "coupon", "cashback", "free shipping",
    "buy now", "limited time", "best price", "lowest price", "ডিসকাউন্ট", "অফার",
    "সেল", "ক্যাশব্যাক", "ফ্রি", "কম দাম", "সস্তা", "বিশেষ", "লিমিটেড",
}

feature_schema = StructType([
    StructField("normalized_review", StringType(), True),
    StructField("safe_review", StringType(), True),
    StructField("model_text", StringType(), True),
    StructField("tokens", ArrayType(StringType()), True),
    StructField("char_length", IntegerType(), True),
    StructField("token_count", IntegerType(), True),
    StructField("bengali_chars", IntegerType(), True),
    StructField("latin_chars", IntegerType(), True),
    StructField("digit_chars", IntegerType(), True),
    StructField("other_chars", IntegerType(), True),
    StructField("codemix_ratio", DoubleType(), True),
    StructField("url_count", IntegerType(), True),
    StructField("email_count", IntegerType(), True),
    StructField("phone_count", IntegerType(), True),
    StructField("price_count", IntegerType(), True),
    StructField("hashtag_count", IntegerType(), True),
    StructField("mention_count", IntegerType(), True),
    StructField("exclamation_count", IntegerType(), True),
    StructField("emoji_count", IntegerType(), True),
    StructField("promo_keyword_hits", IntegerType(), True),
    StructField("repeated_char_ratio", DoubleType(), True),
])


def normalize_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    text = CONTROL_PATTERN.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def redact_text(text):
    text = EMAIL_PATTERN.sub("[EMAIL]", text)
    text = PHONE_PATTERN.sub("[PHONE]", text)
    return URL_PATTERN.sub("[URL]", text)


def strip_model_patterns(text):
    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)
    text = PHONE_PATTERN.sub(" ", text)
    text = PRICE_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def tokenize_text(text, tokenizer=None):
    try:
        raw_tokens = tokenizer.tokenize(text) if tokenizer is not None else TOKEN_PATTERN.findall(text)
    except Exception:
        raw_tokens = TOKEN_PATTERN.findall(text)
    tokens = []
    stopwords = BANGLA_STOPWORDS | ENGLISH_STOPWORDS
    for token in raw_tokens:
        clean_token = str(token).strip().lower()
        if clean_token and TOKEN_PATTERN.fullmatch(clean_token) and clean_token not in stopwords:
            tokens.append(clean_token)
    return tokens


def compute_features(value, tokenizer=None):
    normalized = normalize_text(value)
    safe = redact_text(normalized)
    model_text = strip_model_patterns(normalized)
    tokens = tokenize_text(model_text, tokenizer)

    bengali = len(re.findall(r"[\u0980-\u09ff]", normalized))
    latin = len(re.findall(r"[A-Za-z]", normalized))
    digits = len(re.findall(r"[0-9\u09e6-\u09ef]", normalized))
    other = max(len(normalized) - bengali - latin - digits, 0)
    alphabetic = bengali + latin
    codemix = min(bengali, latin) / alphabetic if alphabetic else 0.0
    repeated = sum(len(match.group(0)) for match in REPEATED_PATTERN.finditer(normalized))

    lower = normalized.lower()
    return (
        normalized,
        safe,
        model_text,
        tokens,
        len(normalized),
        len(tokens),
        bengali,
        latin,
        digits,
        other,
        float(codemix),
        len(URL_PATTERN.findall(normalized)),
        len(EMAIL_PATTERN.findall(normalized)),
        len(PHONE_PATTERN.findall(normalized)),
        len(PRICE_PATTERN.findall(normalized)),
        len(HASHTAG_PATTERN.findall(normalized)),
        len(MENTION_PATTERN.findall(normalized)),
        normalized.count("!"),
        len(emoji.emoji_list(normalized)),
        sum(term in lower for term in PROMOTIONAL_TERMS),
        repeated / max(len(normalized), 1),
    )


@pandas_udf(feature_schema)
def review_features_udf(values: pd.Series) -> pd.DataFrame:
    try:
        from bnlp import BasicTokenizer
        tokenizer = BasicTokenizer()
    except Exception:
        tokenizer = None
    return pd.DataFrame(
        [compute_features(value, tokenizer) for value in values],
        columns=[field.name for field in feature_schema.fields],
    )


def add_review_features(frame):
    enriched = frame.withColumn("_review_features", review_features_udf("review"))
    for field in feature_schema.fields:
        enriched = enriched.withColumn(
            field.name,
            F.col(f"_review_features.{field.name}"),
        )
    enriched = enriched.drop("_review_features")
    return (
        enriched
        .withColumn("has_url", (F.col("url_count") > 0).cast("int"))
        .withColumn("has_email", (F.col("email_count") > 0).cast("int"))
        .withColumn("has_phone", (F.col("phone_count") > 0).cast("int"))
        .withColumn("has_price", (F.col("price_count") > 0).cast("int"))
        .withColumn("has_promo", (F.col("promo_keyword_hits") > 0).cast("int"))
        .withColumn("has_emoji", (F.col("emoji_count") > 0).cast("int"))
    )


In [ ]:
train_features = add_review_features(train)
validation_features = add_review_features(validation)
test_features = add_review_features(test)
banglish_features = add_review_features(banglish)

for name, frame in (
    ("train", train_features),
    ("validation", validation_features),
    ("test", test_features),
    ("banglish", banglish_features),
):
    invalid = frame.filter(
        (F.length("model_text") == 0)
        | (F.col("token_count") == 0)
        | (F.col("char_length") <= 0)
    )
    invalid.write.mode("overwrite").parquet(
        str(PROCESSED_DIR / f"{name}_preprocessing_quarantine.parquet")
    )
    valid = frame.join(invalid.select("review_id"), "review_id", "left_anti")
    globals()[f"{name}_features"] = valid

train_features.write.mode("overwrite").parquet(
    str(PROCESSED_DIR / "train_features.parquet")
)
validation_features.write.mode("overwrite").parquet(
    str(PROCESSED_DIR / "validation_features.parquet")
)
test_features.write.mode("overwrite").parquet(
    str(PROCESSED_DIR / "test_features.parquet")
)
banglish_features.write.mode("overwrite").parquet(
    str(PROCESSED_DIR / "banglish_features.parquet")
)

def length_diagnostics(frame):
    quantiles = frame.approxQuantile("char_length", [0.01, 0.5, 0.95, 0.99], 0.001)
    return {
        "rows": frame.count(),
        "char_length_quantiles": dict(zip(["p01", "p50", "p95", "p99"], quantiles)),
        "maximum_char_length": frame.agg(F.max("char_length")).first()[0],
        "zero_token_rows": frame.filter(F.col("token_count") == 0).count(),
    }

preprocessing_audit = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "stopword_version": STOPWORD_VERSION,
    "gold_train": length_diagnostics(train_features),
    "gold_validation": length_diagnostics(validation_features),
    "gold_test": length_diagnostics(test_features),
    "banglish_sample": length_diagnostics(banglish_features),
    "privacy": {
        "excluded_source_fields": ["Buyer ID", "Images"],
        "redacted_in_safe_review": ["email", "phone", "URL"],
        "model_text_removed_patterns": ["email", "phone", "URL", "price", "mention"],
    },
}
with open(AUDIT_DIR / "preprocessing_audit.json", "w", encoding="utf-8") as handle:
    json.dump(preprocessing_audit, handle, ensure_ascii=False, indent=2)


In [ ]:
gold_eda = train_features.unionByName(validation_features)
gold_eda.createOrReplaceTempView("gold_eda")
banglish_features.createOrReplaceTempView("banglish_eda")

class_distribution = spark.sql("""
    SELECT label, COUNT(*) AS rows,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM gold_eda
    GROUP BY label
    ORDER BY label
""").toPandas()

fig, axis = plt.subplots(figsize=(6, 4))
axis.bar(class_distribution["label"].astype(str), class_distribution["rows"])
axis.set(xlabel="Label", ylabel="Reviews", title="BFRD class distribution")
fig.tight_layout()
fig.savefig(EDA_DIR / "class_distribution.png", dpi=180)
plt.close(fig)

gold_plot = gold_eda.select(
    "label", "char_length", "token_count", "codemix_ratio"
).toPandas()
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
sns.histplot(gold_plot, x="char_length", hue="label", bins=40, ax=axes[0, 0])
sns.boxplot(gold_plot, x="label", y="char_length", ax=axes[0, 1])
sns.histplot(gold_plot, x="token_count", hue="label", bins=40, ax=axes[1, 0])
sns.histplot(gold_plot, x="codemix_ratio", hue="label", bins=30, ax=axes[1, 1])
fig.tight_layout()
fig.savefig(EDA_DIR / "length_token_codemix.png", dpi=180)
plt.close(fig)

signal_columns = [
    "emoji_count", "url_count", "phone_count", "price_count",
    "hashtag_count", "exclamation_count", "promo_keyword_hits",
]
signal_expressions = [
    F.avg(column).alias(column) for column in signal_columns
]
signal_table = (
    gold_eda.groupBy("label")
    .agg(*signal_expressions)
    .orderBy("label")
    .toPandas()
)
signal_long = signal_table.melt(
    id_vars="label", var_name="signal", value_name="mean"
)
fig, axis = plt.subplots(figsize=(11, 5))
sns.barplot(signal_long, x="signal", y="mean", hue="label", ax=axis)
axis.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(EDA_DIR / "handcrafted_signals.png", dpi=180)
plt.close(fig)

duplicate_table = (
    gold_eda.groupBy("duplicate_group_id")
    .agg(
        F.count("*").alias("group_size"),
        F.max("exact_dup_cluster_size").alias("exact_dup_cluster_size"),
    )
    .groupBy("group_size", "exact_dup_cluster_size")
    .count()
    .orderBy("group_size", "exact_dup_cluster_size")
    .toPandas()
)
duplicate_table.to_csv(EDA_DIR / "duplicate_group_distribution.csv", index=False)

source_summary = spark.sql("""
    SELECT 'gold' AS source, label,
           COUNT(*) AS rows,
           AVG(char_length) AS mean_length,
           AVG(token_count) AS mean_tokens,
           AVG(codemix_ratio) AS mean_codemix
    FROM gold_eda
    GROUP BY label
    UNION ALL
    SELECT 'BanglishRev' AS source, NULL AS label,
           COUNT(*) AS rows,
           AVG(char_length) AS mean_length,
           AVG(token_count) AS mean_tokens,
           AVG(codemix_ratio) AS mean_codemix
    FROM banglish_eda
""").toPandas()
source_summary.to_csv(EDA_DIR / "source_summary.csv", index=False)

numeric_columns = [
    "char_length", "token_count", "codemix_ratio", "repeated_char_ratio",
    "url_count", "phone_count", "price_count", "hashtag_count",
    "exclamation_count", "emoji_count", "promo_keyword_hits",
]
correlation = gold_eda.select(*numeric_columns, "label").toPandas().corr()
fig, axis = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation, cmap="vlag", center=0, ax=axis)
fig.tight_layout()
fig.savefig(EDA_DIR / "correlation_heatmap.png", dpi=180)
plt.close(fig)

print(class_distribution.to_string(index=False))
print(source_summary.to_string(index=False))
